# Adversarial Search Strategies and Decision Trees

# Introdução

Este projeto implementa o jogo pop-out, que se trata uma versão ligeiramente diferente do connect 4.

Neste jogo, os jogadores, à vez, colocam peças num tabuleiro vertical com o objetivo de alinhar quatro peças da sua cor, seja na horizontal, vertical ou diagonal. No entanto, ao contrário do connect 4 tradicional, o pop-out game tem pelo menos uma regra diferente: possibilidade de remover uma peça da base de uma coluna, desde que essa peça pertença ao jogador atual, o que aumenta significativamente a complexidade do jogo, tornando-o mais dinâmico e imprevisível.

Com esta nova regra, dá-se um aumento da complexidade e estratégias determinísticas tornam-se difíceis de aplicar. Assim, neste projeto, optou-se pela utilização de métodos baseados em simulação, mais concretamente algoritmos de Monte Carlo, que permitem tomar decisões aproximadas através da execução de múltiplas simulações aleatórias do jogo.

O principal objetivo do projeto é então desenvolver uma representação do estado do jogo adequada para simulações automáticas e implementar diferentes versões do algoritmo de Monte Carlo para selecionar jogadas. Além disso, tem se como objetivo analisar o desempenho destas abordagens e explorar a geração de dados que possam ser utilizados posteriormente em modelos de aprendizagem automática, como árvores de decisão.

Este notebook apresenta a implementação do jogo, a construção do estado para simulação, a aplicação de várias versões do algoritmo de Monte Carlo e do algoritmo ID3 das árvores de decisão e a análise dos resultados obtidos.




# Jogo

## Modos de jogo:

  - pessoa vs pessoa;
  
  - computador vs pessoa; 
  
  - computador vs computador;

É possível escolher o modo de jogo antes de começar a jogar, ver os diferentes comandos que existem para podermos jogar e as regras do jogo  no menu inicial.

## Jogadas: 

Existem essencialmente três tipos de jogadas possíveis: inserir e remover uma peça e declarar empate. 

   - **Inserir peça (I)**: colocar uma peça no topo de uma coluna, que depois “cai” até à posição disponível mais baixa.
  
   - **Remover peça (R)**: retirar a peça da base de uma coluna, fazendo com que todas as peças acima desçam uma posição no tabuleiro.
   
   - **Empate (D)**: declarar empate, quando permitido pelas regras abaixo.

## Regras importantes: 

- Se uma jogada de remoção ("pop") criar quatro em linha para os dois jogadores, vence o jogador que realizou essa jogada.

- Se o tabuleiro estiver cheio, o jogador pode escolher entre remover uma peça ou declarar empate.

- Se o mesmo estado do jogo ocorrer três vezes, qualquer jogador pode declarar empate.

## Implementação:

Para já o jogo encontra-se implementado em dois ficheiros .py, "pop_out2.game.py" que contém a versão iterativa do jogo, ou seja, a versão que permite jogar jogador vs jogador, e o ficheiro "popout_state.py" que implementa a versão do jogo que pode ser usada pelos algoritmos de intelegência artificial que vamos implememtar (MCT e ID3), ou seja, a versão computador vs computador do jogo.
O ficheiro main.py contém o motor do jogo e está ligado a estes dois ficheiros, ou seja, estamos a utilizar o mesmo motor de jogo para os dois modos de jogo. Ao contrário do modo pessoa vs pessoa, o modo computador vs computador não requere a iteração com o usuário por isso é que criamos uma nova class do jogo que não realiza chamadas de input.

Para poder correr o jogo devemos usar o comando python main.py no terminal.

(Falta criar e falar do modo computador vs pessoa e não sei conseguimos juntar popout2 e popout_state)




  
    

# Pop_out2.py

Modo jogo jogador vs jogador

In [ ]:
import os
import random
import time

def new_board():
    b = []
    for i in range(0,6): b += [["."]*7]
    return b

def start_player(rand):
    if (rand == False): return "O"
    else: 
        player = ["O","X"]
        return player[random.randint(0,1)]

def clear_terminal():
    os.system('cls' if os.name == 'nt' else 'clear')

def check(line, char):
    count = 0
    for cell in line:
        if cell == char:
            count += 1
            if count == 4:
                return True
        else:
            count = 0
    return False

class game:
    def __init__(self, rand):
        self.board = new_board()
        self.board_history = {self.board_to_string(): 1}
        self.cur_player = start_player(rand)
        self.running = True

    def change_player(self):
        if (self.cur_player == "O"): self.cur_player = "X"
        else: self.cur_player = "O"
        return
    
    def print_board(self):
        print(self.board_to_string())
        return

    def board_to_string(self):
        s = ""
        for row in range(0, len(self.board)):
            for col in range(0, len(self.board[row])):
                s += self.board[row][col] + " "
            s += "\n"
        return s

    def board_is_full(self):
        for col in self.board[0]:
            if (col == "."): return False
        return True
    
    def check_for_win(self, move_type="insert"):
        o_wins = False
        x_wins = False

        for row in range(len(self.board)):
            if check(self.board[row], "O"): o_wins = True
            if check(self.board[row], "X"): x_wins = True

        for col in range(len(self.board[0])):
            cur_col = []
            for row in range(len(self.board)):
                cur_col.append(self.board[row][col])
            if check(cur_col, "O"): o_wins = True
            if check(cur_col, "X"): x_wins = True

        for row in range(len(self.board) - 3):
            for col in range(len(self.board[0]) - 3):
                diag = []
                for i in range(4):
                    diag.append(self.board[row + i][col + i])
                if check(diag, "O"): o_wins = True
                if check(diag, "X"): x_wins = True

        for row in range(3, len(self.board)):
            for col in range(len(self.board[0]) - 3):
                diag = []
                for i in range(4):
                    diag.append(self.board[row - i][col + i])
                if check(diag, "O"): o_wins = True
                if check(diag, "X"): x_wins = True

        if not o_wins and not x_wins:
            return False

        if move_type == "pop" and o_wins and x_wins:
            self.end_game(1)
            return True

        if self.cur_player == "O" and o_wins:
            self.end_game(1)
            return True

        if self.cur_player == "X" and x_wins:
            self.end_game(1)
            return True

        self.end_game(-1)
        return True

    def invalid_command(self):
        clear_terminal()
        self.print_board()
        print("Invalid command. To check command rules, use COMMANDS.")
        print("Use ENTER to PLAY AGAIN")
        input()
        return
    
    def invalid_move(self):
        clear_terminal()
        self.print_board()
        print("Invalid move. To check game rules, use RULES")
        print("Use ENTER to PLAY AGAIN")
        input()
        return

    def check_commands(self):
        clear_terminal()
        print("To remove a disc from the bottom row, use R followed by the column index between 1 and 7.")
        print("For example, R5 removes the bottom disc from the 5th column.")
        print("NOTE: The disc is only removable if it belongs to the current player.")
        print()
        print("To insert a disc, use I followed by the column index between 1 and 7.")
        print("For example, I3 inserts a disc in the top of the 3rd column.")
        print("NOTE: The disc can only be inserted if the column is not full yet.")
        print()
        print("To declare a draw, use D.")
        print()
        print("To restart game, use RESTART")
        print()
        print("To end game and return to menu, use QUIT")
        print()
        print("Press ENTER to return")
        input()
        return

    def check_rules(self):
        clear_terminal()
        print("POP OUT is a version of CONNECT 4 with some changes.")
        print()
        print("The game starts with an empty 7*6 board. Players alternate turns placing their own coloured discs into the board.")
        print("A player, in their round, can either add another disc from the top or remove a disc of one's own colour from the bottom.")
        print("The latter will drop each disc above it down one space.")
        print("The first player to connect four of their discs horizontally, vertically or diagonally wins the game.")
        print()
        print("ADDITIONAL RULES:")
        print("1. If a pop move creates four-in-rows for both players, the player who made the pop move is the winner.")
        print("2. If the board is full, the player to move decides whether he wants to make a pop move or end the game as a draw.")
        print("3. If the same state is repeated three times, either player can declare the game drawn.")
        print()
        print("Press ENTER to return")
        input()
        return
    
    def end_game(self, result):
        clear_terminal()
        self.print_board()
        if (result == 1): print("Player " + self.cur_player + " is the winner!")
        elif (result == 0): print("It's a draw!")
        else:
            self.change_player()
            print("Player " + self.cur_player + " is the winner!")
        print()
        print("Use RESTART or QUIT")
        while(True):
            command = input()
            if (command == "RESTART"): 
                self.restart(False)
                return
            elif (command == "QUIT"):
                self.running = False
                return

    def remove(self, col):
        try: col = int(col)
        except:
            self.invalid_move()
            return                          
        if (col < 1 or col > 7):
            self.invalid_move()
            return                          
        elif (self.board[len(self.board) - 1][col - 1] == self.cur_player): 
            for i in range(len(self.board) - 1, 0, -1):
                self.board[i][col - 1] = self.board[i - 1][col - 1]
            self.board[0][col - 1] = "."
            if (self.board_to_string() in self.board_history.keys()): self.board_history[self.board_to_string()] += 1
            else: self.board_history[self.board_to_string()] = 1
            if (self.check_for_win("pop")): return
            self.change_player()
            return
        else: self.invalid_move()
        return
    
    def insert(self, col):
        try: col = int(col)
        except:
            self.invalid_move()
            return                         
        if (col < 1 or col > 7):
            self.invalid_move()
            return                         
        elif (self.board[0][col - 1] == "."):
            self.board[0][col - 1] = self.cur_player
            i = 0
            clear_terminal()
            self.print_board()
            while (i < 5 and self.board[i+1][col - 1] == "."):
                time.sleep(0.25)
                self.board[i][col - 1] = "."
                self.board[i+1][col - 1] = self.cur_player
                i += 1
                clear_terminal()
                self.print_board()
            if (self.board_to_string() in self.board_history.keys()): self.board_history[self.board_to_string()] += 1
            else: self.board_history[self.board_to_string()] = 1
            time.sleep(0.5)
            if (self.check_for_win("insert")): return
            self.change_player()
            return
        else: self.invalid_move()
        return
    
    def draw(self):
        if self.board_is_full():
            self.end_game(0)
        elif self.board_history[self.board_to_string()] >= 3:
            self.end_game(0)
        else:
            self.invalid_move()
        return

    def restart(self, rand):
        self.__init__(rand)
        self.make_a_move()
        return
        
    def make_a_move(self):
        self.running = True

        while self.running:
            clear_terminal()
            self.print_board()
            print("PLAYING NOW: " + self.cur_player)

            command = input().strip()

            if command.upper() == "COMMANDS":
                self.check_commands()
                continue
            elif command.upper() == "RULES":
                self.check_rules()
                continue
            elif command.upper() == "RESTART":
                self.restart(False)
                return
            elif command.upper() == "QUIT":
                self.running = False
                return
            elif len(command) == 0:
                continue
            elif (command[0].upper() == "R") and len(command) == 2:
                self.remove(command[1])
            elif (command[0].upper() == "I") and len(command) == 2:
                self.insert(command[1])
            elif command.upper() == "D":
                self.draw()
            else:
                self.invalid_command()

# Popout_state.py

Modo jogo computador vs computador

In [1]:
#versão em que o computador pode jogar
#futuramente temos de fazer conexão com o play.py
#basicamente é igual ao popout.py mas sem pedidos de input

import copy

def new_board():
    return [["."] * 7 for _ in range(6)]


def check(line, char):
    count = 0
    for cell in line:
        if cell == char:
            count += 1
            if count == 4:
                return True
        else:
            count = 0
    return False


class PopOutState:
    def __init__(self, board=None, current_player="O", board_history=None, winner=None, terminal=False):
        self.board = copy.deepcopy(board) if board is not None else new_board()
        self.current_player = current_player
        self.board_history = copy.deepcopy(board_history) if board_history is not None else {}
        self.winner = winner
        self.terminal = terminal

        if not self.board_history:
            self.board_history[self.board_to_string()] = 1

    def clone(self):
        return PopOutState(
            board=self.board,
            current_player=self.current_player,
            board_history=self.board_history,
            winner=self.winner,
            terminal=self.terminal
        )

    def board_to_string(self):
        s = ""
        for row in self.board:
            s += " ".join(row) + "\n"
        return s

    def change_player(self):
        self.current_player = "X" if self.current_player == "O" else "O"

    def board_is_full(self):
        return all(cell != "." for cell in self.board[0])

    def get_valid_moves(self):
        moves = []

        # Insert moves
        for col in range(7):
            if self.board[0][col] == ".":
                moves.append(("I", col + 1))

        # Remove moves
        for col in range(7):
            if self.board[5][col] == self.current_player:
                moves.append(("R", col + 1))

        # Optional draw declaration
        if self.board_is_full() or self.board_history.get(self.board_to_string(), 0) >= 3:

            moves.append(("D", None))

        return moves

    def has_four(self, player):
        # Rows
        for row in range(6):
            if check(self.board[row], player):
                return True

        # Columns
        for col in range(7):
            cur_col = []
            for row in range(6):
                cur_col.append(self.board[row][col])
            if check(cur_col, player):
                return True

        # Diagonal \
        for row in range(6 - 3):
            for col in range(7 - 3):
                diag = []
                for i in range(4):
                    diag.append(self.board[row + i][col + i])
                if check(diag, player):
                    return True

        # Diagonal /
        for row in range(3, 6):
            for col in range(7 - 3):
                diag = []
                for i in range(4):
                    diag.append(self.board[row - i][col + i])
                if check(diag, player):
                    return True

        return False

    def update_terminal_status(self, move_type):
        o_wins = self.has_four("O")
        x_wins = self.has_four("X")

        if not o_wins and not x_wins:
            return

        if move_type == "pop" and o_wins and x_wins:
            self.winner = self.current_player
            self.terminal = True
            return

        if self.current_player == "O" and o_wins:
            self.winner = "O"
            self.terminal = True
            return

        if self.current_player == "X" and x_wins:
            self.winner = "X"
            self.terminal = True
            return

        self.winner = "X" if self.current_player == "O" else "O"
        self.terminal = True

    def register_state(self):
        key = self.board_to_string()
        if key in self.board_history:
            self.board_history[key] += 1
        else:
            self.board_history[key] = 1

    def apply_insert(self, col):
        c = col - 1

        if c < 0 or c > 6:
            raise ValueError("Invalid column")

        if self.board[0][c] != ".":
            raise ValueError("Column is full")

        self.board[0][c] = self.current_player

        row = 0
        while row < 5 and self.board[row + 1][c] == ".":
            self.board[row][c] = "."
            self.board[row + 1][c] = self.current_player
            row += 1

        self.update_terminal_status("insert")
        if not self.terminal:
            self.change_player()
            self.register_state()

    def apply_remove(self, col):
        c = col - 1

        if c < 0 or c > 6:
            raise ValueError("Invalid column")

        if self.board[5][c] != self.current_player:
            raise ValueError("Bottom disc does not belong to current player")

        for row in range(5, 0, -1):
            self.board[row][c] = self.board[row - 1][c]

        self.board[0][c] = "."

        self.update_terminal_status("pop")
        if not self.terminal:
            self.change_player()
            self.register_state()

    def apply_move(self, move):
        if self.terminal:
            raise ValueError("Game is already over")

        kind, value = move

        if kind == "I":
            self.apply_insert(value)
        elif kind == "R":
            self.apply_remove(value)
        elif kind == "D":
            if self.board_is_full() or self.board_history.get(self.board_to_string(), 0) >= 3:
                self.winner = None
                self.terminal = True
            else:
                raise ValueError("Draw declaration is not valid in this state")
        else:
            raise ValueError("Unknown move type")

    def is_terminal(self):
        return self.terminal

    def get_result(self, player):
        if not self.terminal:
            raise ValueError("Game is not over yet")

        if self.winner is None:
            return 0.5
        elif self.winner == player:
            return 1.0
        else:
            return 0.0


# Main.py

É o motor do jogo, neste momento não é poossível correr tudo diretamente no notebook, pelo que devemos usar o comando python main.py no terminal.

In [ ]:
from pop_out2 import game, clear_terminal
from popout_state import PopOutState
from montecarlo import monte_carlo_move

import sys
import time

while True:
    clear_terminal()
    print("Welcome to POP OUT!")
    print()
    print("1. CHECK COMMANDS\n2. CHECK RULES\n3. PLAY\n4. EXIT")
    match(input().strip()):
        case "1":
            clear_terminal()
            print("To remove a disc from the bottom row, use R followed by the column index between 1 and 7.")
            print("For example, R5 removes the bottom disc from the 5th column.")
            print("NOTE: The disc is only removable if it belongs to the current player.")
            print()
            print("To insert a disc, use I followed by the column index between 1 and 7.")
            print("For example, I3 inserts a disc in the top of the 3rd column.")
            print("NOTE: The disc can only be inserted if the column is not full yet.")
            print()
            print("To declare a draw, use D.")
            print()
            print("To restart game, use RESTART")
            print()
            print("To end game and return to menu, use QUIT")
            print()
            print("Press ENTER to return")
            input()
        case "2":
            clear_terminal()
            print("POP OUT is a version of CONNECT 4 with some changes.")
            print()
            print("The game starts with an empty 7*6 board. Players alternate turns placing their own coloured discs into the board.")
            print("A player, in their round, can either add another disc from the top or remove a disc of one's own colour from the bottom.")
            print("The latter will drop each disc above it down one space.")
            print("The first player to connect four of their discs horizontally, vertically or diagonally wins the game.")
            print()
            print("ADDITIONAL RULES:")
            print("1. If a pop move creates four-in-rows for both players, the player who made the pop move is the winner.")
            print("2. If the board is full, the player to move decides whether he wants to make a pop move or end the game as a draw.")
            print("3. If the same state is repeated three times, either player can declare the game drawn.")
            print()
            print("Press ENTER to return")
            input()
        case "3":
            while True:
                clear_terminal()
                print("Select game mode:")
                print()
                print("1. PLAYER VS PLAYER\n2. PLAYER VS COMPUTER\n3. COMPUTER VS COMPUTER\n4. RETURN TO MENU")
                match(input().strip()):
                    case "1":
                        clear_terminal()
                        game1 = game(False)
                        game1.make_a_move()
                    case "2":
                        clear_terminal()
                        print("Under construction...")
                        time.sleep(2)
                    case "3":
                        clear_terminal()
                        state = PopOutState()
              
                        while not state.is_terminal():
                            clear_terminal()
                            print(state.board_to_string())
                            print("Current player:", state.current_player)
                            move = monte_carlo_move(state, simulations_per_move=20)
                            print("Computer plays:", move)
                            time.sleep(1.5)
                            state.apply_move(move)
                            time.sleep(1.5)
                            print("Board after move:")
                            print(state.board_to_string())
                            time.sleep(2)

                            

                        clear_terminal()
                        print(state.board_to_string())

                        if state.winner is None:
                         print("It's a draw!")
                        else:
                            print("Winner:", state.winner)

                        input("Press ENTER to return")


                    case "4":
                        break
                    case _:
                        clear_terminal()
                        print("Invalid command! Returning to GAME SELECTION...")
                        time.sleep(2)
        case "4":
            sys.exit("Goodbye!")
        case _:
            clear_terminal()
            print("Invalid command! Returning to MENU...")
            time.sleep(2)

# Monte Carlo


(Várias versões do monte carlo e análise comparação de cada versão)
(criar data-set do jogo)

Nesta secção implementamos várias versões do algoritmo Monte Carlo e vamos analisar o poder de tomadas de decisão em relação às jogadas que cada versão tem

## Teoria do Monte Carlo:

O Monte Carlo baseia-se na execução de múltiplas simulações aleatórias (playouts) a partir de um determinado estado do jogo. Para cada jogada possível:

1. Aplica-se a jogada ao estado atual;

2. Simula-se o resto da partida de forma aleatória;

3. Regista-se o resultado final (vitória, derrota ou empate);

4. Repete-se o processo várias vezes;

5. Calcula-se a média dos resultados.

A jogada escolhida será aquela que apresentar melhor resulado de média.




## Versão 1 (montecarlo.py)

Começamos por implementar um algoritmo de Monte Carlo simples (também conhecido como *Flat Monte Carlo*), cujo objetivo é  apenas selecionar a melhor jogada com base em simulações aleatórias do jogo. Basicamnete, para cada jogada possível no estado atual, o algoritmo executa várias simulações completas onde, após a jogada inicial, as restantes jogadas são escolhidas aleatoriamente até ao final da partida. Cada uma destas simulações resulta num resultado final (vitória, derrota ou empate), que é avaliado do ponto de vista do jogador atual.
Os resultados obtidos em todas as simulações são então agregados, sendo calculada uma pontuação média para cada jogada. Esta pontuação reflete a qualidade esperada da jogada com base nas simulações realizadas. Por fim, o algoritmo seleciona a jogada que apresenta a melhor pontuação média, assumindo que esta tem maior probabilidade de conduzir a um resultado favorável.
Embora seja uma versão simples, esta permite tomar decisões mais informadas do que uma estratégia puramente aleatória. No entanto, apresenta algumas limitações, nomeadamente o facto de não reutilizar informação entre simulações e depender fortemente do número de simulações realizadas, o que pode impactar o tempo de execução.

In [3]:
#implementaçao de um monte carlo simples
#usa as regras do popout_state

import random

#permite fazer uma jogada aleatoria
def random_move(state):
    return random.choice(state.get_valid_moves())

#obter o resultado dessa jogada
def random_playout(state, root_player):
    simulation_state = state.clone()

    while not simulation_state.is_terminal():
        move = random.choice(simulation_state.get_valid_moves())
        simulation_state.apply_move(move)

    return simulation_state.get_result(root_player)

#algoritmo de montecarlo
def monte_carlo_move(state, simulations_per_move=50):
    legal_moves = state.get_valid_moves()
    root_player = state.current_player

    best_move = None
    best_score = -1

    for move in legal_moves:
        total_score = 0

        for _ in range(simulations_per_move):
            sim = state.clone()
            sim.apply_move(move)
            total_score += random_playout(sim, root_player)  # ← substituição

        average_score = total_score / simulations_per_move

        if average_score > best_score:
            best_score = average_score
            best_move = move

    return best_move

## Versão 2

Implementamos agora uma versão que contém uma heurística que serve de guia às simulações, ou seja, ao contrário da versão anterior as jogadas aleatórias dão preferência a um jogador.


In [4]:
def heuristic_guided_move(state):
    # Se houver jogada que ganhe imediatamente → faz
    for move in state.get_valid_moves():
        clone = state.clone()
        clone.apply_move(move)
        if clone.winner == state.current_player:
            return move
    # Caso contrário, evita jogadas ruins
    safe_moves = []
    for move in state.get_valid_moves():
        clone = state.clone()
        clone.apply_move(move)
        # evita jogadas que dão vitória imediata ao oponente
        for opp_move in clone.get_valid_moves():
            temp = clone.clone()
            temp.apply_move(opp_move)
            if temp.winner == clone.current_player:
                break
        else:  # nenhum contra-ataque imediato
            safe_moves.append(move)
    if safe_moves:
        return random.choice(safe_moves)
    else:
        return random.choice(state.get_valid_moves())
def monte_carlo_move_v2(state, simulations_per_move=50):
    legal_moves = state.get_valid_moves()
    root_player = state.current_player

    best_move = None
    best_score = -1

    for move in legal_moves:
        total_score = 0

        for _ in range(simulations_per_move):
            sim = state.clone()
            sim.apply_move(move)

            while not sim.is_terminal():
                sim.apply_move(heuristic_guided_move(sim))  # ← diferença da V1

            total_score += sim.get_result(root_player)

        average_score = total_score / simulations_per_move

        if average_score > best_score:
            best_score = average_score
            best_move = move

    return best_move  


## Versão 3



## Criação do data-set

Geramos o data-set do jogo, deixando as várias versões do algoritmo MCT realizar vários jogos, escolhemos os jogos que faziam mais sentido em termos de jogadas e resultados finais e evitamos jogos repetidos. Depois disso guardamos como data-set as todas as jogadas e estados do tabuleiro para cada um desses jogos que selecionamos anteriormente. (Não tenho a certeza acho que foi assim que o professor explicou)



# Árvores de decisão

(para o iris data-set e o do jogo)

In [10]:
import pandas as pd

df = pd.read_csv("iris.csv")



# Conclusão